# 4.5 Prefix Caching Lab[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.5_prefix_caching/lab.ipynb)[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.5_prefix_caching/lab.ipynb)Benchmark prefix caching: measure TTFT with and without KV reuse for shared system prompts.

In [ ]:
# ============================================================# Setup: clone repo utilities, install dependencies# ============================================================import subprocess, sys, os# Clone repo for shared utilities (persists across cells)if not os.path.exists("/tmp/lis-repo"):    # Process this step    subprocess.run(["git", "clone", "--depth=1",                    "https://github.com/harshuljain13/llm-inference-at-scale.git",                    "/tmp/lis-repo"], check=True)sys.path.insert(0, "/tmp/lis-repo")# Install vLLM (required for prefix caching benchmark)subprocess.run([sys.executable, "-m", "pip", "install", "-q",                "vllm>=0.4.0", "matplotlib", "numpy"], check=True)

## Experiment 1: Manual KV Reuse SimulationWe simulate prefix caching savings by measuring prefill time for the full sequence vs. suffix-only.This works on any GPU (including Colab T4) using HuggingFace transformers.

In [ ]:
# ============================================================# Parameters: adjust these to change the experiment# ============================================================# Non-gated model, no auth required, GQA architectureMODEL_NAME = "mistralai/Mistral-7B-v0.1"  # Non-gated, no auth needed# Length of shared prefix to cache (longer = more savings)SYSTEM_PROMPT_TOKENS = 512   # Length of simulated system prompt# Length of unique user query appended after prefixUSER_QUERY_TOKENS = 64       # Length of simulated user query# Repetitions per measurement for timing stabilityNUM_TRIALS = 5               # Repetitions for timing stability# Target device for inferenceDEVICE = "cuda"              # Use "cpu" if no GPU available

In [ ]:
# PyTorch: GPU tensor operations and synchronizationimport torch# perf_counter: high-resolution wall-clock timingimport time# NumPy: statistical analysis of timing measurementsimport numpy as np# HuggingFace: model loading with KV cache supportfrom transformers import AutoModelForCausalLM, AutoTokenizer# Load model and tokenizer (downloads ~14GB on first run)print(f"Loading {MODEL_NAME}...")# Load tokenizer for converting text to token IDstokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)# Load in FP16 (~14GB) with automatic GPU layer placementmodel = AutoModelForCausalLM.from_pretrained(    # Process this step    MODEL_NAME,    # Process this step    torch_dtype=torch.float16,    # Process this step    device_map="auto",       # Automatic GPU placement    use_cache=True           # Enable KV cache returns)# Eval mode: disable dropout for deterministic KV cache behaviormodel.eval()print(f"Model loaded on {next(model.parameters()).device}")

In [ ]:
# ============================================================# Generate synthetic input: system prompt + user query# ============================================================# Use real tokens from tokenizer vocabulary# Get vocabulary size to generate valid random token IDsvocab_size = tokenizer.vocab_size# Fixed seed for reproducible synthetic inputstorch.manual_seed(42)# Create fixed system prompt tokens and varying user query tokens# Shared system prompt tokens (same for all requests)system_ids = torch.randint(100, vocab_size - 100, (1, SYSTEM_PROMPT_TOKENS)).to(DEVICE)# Unique user query tokens (different per request)user_ids = torch.randint(100, vocab_size - 100, (1, USER_QUERY_TOKENS)).to(DEVICE)# Full input = system prompt + user query concatenatedfull_ids = torch.cat([system_ids, user_ids], dim=1)print(f"System prompt: {SYSTEM_PROMPT_TOKENS} tokens")print(f"User query: {USER_QUERY_TOKENS} tokens")print(f"Total input: {full_ids.shape[1]} tokens")

In [ ]:
# ============================================================# Benchmark: Full prefill (no caching) vs suffix-only (with cached prefix)# ============================================================# Utility: measure average prefill latency over N trialsdef measure_prefill_time(input_ids, past_key_values=None, num_trials=5):    """Measure average prefill time over multiple trials."""    # Initialize results collection    times = []    # Iterate over each item in the collection    for _ in range(num_trials):        # Synchronize GPU to ensure accurate timing        torch.cuda.synchronize()        # High-resolution timer for benchmarking        start = time.perf_counter()        # Disable gradient tracking for inference        with torch.no_grad():            # Process this step            outputs = model(                # Process this step                input_ids=input_ids,                # Process this step                past_key_values=past_key_values,                # Process this step                use_cache=True            )        torch.cuda.synchronize()        times.append(time.perf_counter() - start)    return np.array(times), outputs# Warmup: first run includes CUDA kernel JIT compilation overhead (first run includes CUDA kernel compilation overhead)_ = measure_prefill_time(full_ids, num_trials=1)# Baseline: process entire sequence (system + query) from scratch: system + user from scratchfull_times, _ = measure_prefill_time(full_ids, num_trials=NUM_TRIALS)# Pre-compute prefix KV cache (simulates LMCache behavior) (simulates cached prefix)with torch.no_grad():    prefix_out = model(input_ids=system_ids, use_cache=True)cached_kv = prefix_out.past_key_values  # This is the "cached" prefix# Cached path: only process user query tokens, reuse prefix KV: only user query, reusing cached KVsuffix_times, _ = measure_prefill_time(user_ids, past_key_values=cached_kv, num_trials=NUM_TRIALS)print(f"Full prefill ({full_ids.shape[1]} tokens):    {full_times.mean()*1000:.1f} ms (std {full_times.std()*1000:.1f} ms)")print(f"Suffix only ({USER_QUERY_TOKENS} tokens):     {suffix_times.mean()*1000:.1f} ms (std {suffix_times.std()*1000:.1f} ms)")print(f"TTFT speedup:                   {full_times.mean()/suffix_times.mean():.2f}x")print(f"Time saved per request:          {(full_times.mean()-suffix_times.mean())*1000:.1f} ms")

## Experiment 2: Scaling Prefix LengthMeasure how TTFT improvement scales with the length of the cached prefix.

In [ ]:
# ============================================================# Sweep prefix lengths: measure speedup at each# ============================================================import matplotlib.pyplot as plt# Sweep prefix lengths from short (128) to long (2048) tokensprefix_lengths = [128, 256, 512, 1024, 2048]# Fixed suffix length (user query portion)suffix_len = 64  # Fixed user query length# Collect speedup metrics for each prefix lengthspeedups = []# Absolute TTFT values for plottingfull_ttfts = []# Initialize results collectioncached_ttfts = []# Iterate over each item in the collectionfor plen in prefix_lengths:    # Generate input    p_ids = torch.randint(100, vocab_size - 100, (1, plen)).to(DEVICE)    # Create tensor for computation    s_ids = torch.randint(100, vocab_size - 100, (1, suffix_len)).to(DEVICE)    # Create tensor for computation    combined = torch.cat([p_ids, s_ids], dim=1)    # Full prefill    torch.cuda.synchronize()    # High-resolution timer for benchmarking    t0 = time.perf_counter()    # Disable gradient tracking for inference    with torch.no_grad():        # Process this step        model(input_ids=combined, use_cache=True)    # Synchronize GPU to ensure accurate timing    torch.cuda.synchronize()    # High-resolution timer for benchmarking    full_t = time.perf_counter() - t0    # Cached prefix + suffix only    with torch.no_grad():        # Process this step        pout = model(input_ids=p_ids, use_cache=True)    # Synchronize GPU to ensure accurate timing    torch.cuda.synchronize()    t0 = time.perf_counter()    with torch.no_grad():        model(input_ids=s_ids, past_key_values=pout.past_key_values, use_cache=True)    torch.cuda.synchronize()    suffix_t = time.perf_counter() - t0    speedups.append(full_t / suffix_t)    full_ttfts.append(full_t * 1000)    cached_ttfts.append(suffix_t * 1000)    print(f"Prefix {plen:>5} tokens: full={full_t*1000:.1f}ms, cached={suffix_t*1000:.1f}ms, speedup={full_t/suffix_t:.2f}x")

In [ ]:
# ============================================================# Plot: TTFT with and without prefix caching# ============================================================fig_prefix, (ax_ttft, ax_speedup) = plt.subplots(1, 2, figsize=(12, 5))# Left: absolute TTFT comparison# X positions for bar chart (one per prefix length)x = range(len(prefix_lengths))# Bar width for side-by-side comparisonwidth = 0.35# Draw bar chart for this metricax_ttft.bar([i - width/2 for i in x], full_ttfts, width, label="No Cache (full prefill)", color="#ffe4e6", edgecolor="#000")# Draw bar chart for this metricax_ttft.bar([i + width/2 for i in x], cached_ttfts, width, label="With Prefix Cache", color="#dcfce7", edgecolor="#000")ax_ttft.set_xlabel("Cached Prefix Length (tokens)")ax_ttft.set_ylabel("TTFT (ms)")ax_ttft.set_title("Time-to-First-Token: Full vs Cached Prefill")# Process this stepax_ttft.set_xticks(x)# Process this stepax_ttft.set_xticklabels(prefix_lengths)ax_ttft.legend()ax_ttft.grid(axis="y", alpha=0.3)# Right: speedup factorax_speedup.plot(prefix_lengths, speedups, "o-", color="#2563eb", linewidth=2, markersize=8)# Reference line for comparison thresholdax_speedup.axhline(y=1, color="#991b1b", linestyle="--", alpha=0.5, label="No improvement")ax_speedup.set_xlabel("Cached Prefix Length (tokens)")ax_speedup.set_ylabel("TTFT Speedup (x)")ax_speedup.set_title("Speedup from Prefix Caching")ax_speedup.legend()ax_speedup.grid(alpha=0.3)# Adjust spacing to prevent label overlapplt.tight_layout()# Save figure to disk for referenceplt.savefig("prefix_caching_benchmark.png", dpi=150, bbox_inches="tight")# Render the chartplt.show()# Process this stepprint("Saved: prefix_caching_benchmark.png")

## Experiment 3: vLLM Prefix Caching (requires A10G+ GPU)If running on a GPU with 24GB+ VRAM, this cell benchmarks vLLM's automatic prefix caching end-to-end.

In [ ]:
# ============================================================# vLLM prefix caching benchmark (skip if insufficient VRAM)# ============================================================try:    from vllm import LLM, SamplingParams    # Check available VRAM    free_mem = torch.cuda.mem_get_info()[0] / 1e9    # Conditional check    if free_mem < 16:        # Display formatted result        print(f"Only {free_mem:.1f} GB free VRAM. Need 16GB+ for vLLM. Skipping.")        # Process this step        raise MemoryError("Insufficient VRAM")    # Delete HF model to free memory    del model    # Release GPU memory cache back to system    torch.cuda.empty_cache()    # System prompt shared across all requests    SYSTEM_PROMPT = "You are a helpful AI assistant. " * 200  # ~800 tokens    # Launch vLLM WITH prefix caching    llm_cached = LLM(        # Process this step        model="mistralai/Mistral-7B-v0.1",        # Process this step        enable_prefix_caching=True,        # Process this step        gpu_memory_utilization=0.85,        # Process this step        max_model_len=4096    # Process this step    )    # Process this step    params = SamplingParams(max_tokens=1, temperature=0)  # 1 token to measure TTFT    # Generate requests with shared prefix    prompts = [SYSTEM_PROMPT + f"Question {i}: What is {i}+{i}?" for pi in range(20)]    # First batch: cold cache    t0 = time.perf_counter()    # Run model generation (prefill + decode)    llm_cached.generate(prompts[:5], params)    cold_time = (time.perf_counter() - t0) / 5    # Second batch: warm cache (prefix already computed)    t0 = time.perf_counter()    llm_cached.generate(prompts[5:10], params)    warm_time = (time.perf_counter() - t0) / 5    print(f"Cold cache TTFT (per request): {cold_time*1000:.1f} ms")    print(f"Warm cache TTFT (per request): {warm_time*1000:.1f} ms")    print(f"Speedup from prefix caching:   {cold_time/warm_time:.2f}x")except (ImportError, MemoryError, Exception) as e:    print(f"vLLM benchmark skipped: {e}")    print("Run on A10G/A100 for full vLLM prefix caching benchmark.")

## Key Takeaways1. **Prefix caching is memoization for attention**: identical token prefixes produce identical KV tensors.2. **TTFT scales with cached fraction**: 80% cached prefix = ~80% TTFT reduction.3. **Memory savings are Nx** where N = concurrent requests sharing the same prefix.4. **Exact match required**: one different token breaks the entire cache chain downstream.5. **Design prompts for cacheability**: static content first, dynamic content last.